# VisionGuard AI — Model Training

This notebook trains and evaluates both VisionGuard models on **MVTec AD** and exports them for the web application.

| Stage | Model | Task |
|---|---|---|
| 1 | YOLOv8s-seg | Pixel-level defect segmentation (single class: `defect`) |
| 2 | YOLO11s-cls | 15-class product classification |

**Kaggle settings:** Accelerator = GPU (T4 ×2 recommended), Internet = On.

**Output:** `/kaggle/working/export/` containing `defect_segmentation.pt`, `product_classifier.pt` and `model_config.json`
(also zipped as `visionguard_models.zip`). Copy these three files into `backend/model/`.

## 1. Setup

In [ ]:
!pip install -q ultralytics kagglehub scikit-learn

import torch

n_gpu = torch.cuda.device_count()
print("CUDA available:", torch.cuda.is_available())
for i in range(n_gpu):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

# Use two GPUs when available, otherwise one GPU, otherwise CPU
DEVICE = [0, 1] if n_gpu >= 2 else (0 if n_gpu == 1 else "cpu")
print("Training device:", DEVICE)

## 2. Dataset — MVTec AD

In [ ]:
import kagglehub
from pathlib import Path

download_path = Path(kagglehub.dataset_download("ipythonx/mvtec-ad"))

# Locate the folder that directly contains the 15 category folders
MVTEC_PATH = next(p.parent for p in download_path.rglob("bottle") if p.is_dir() and (p / "test").exists())

categories = sorted(p.name for p in MVTEC_PATH.iterdir() if p.is_dir() and (p / "test").exists())
print("MVTec path :", MVTEC_PATH)
print("Categories :", len(categories))
print(categories)
assert len(categories) == 15

## 3. Build the segmentation dataset

Binary ground-truth masks are converted to YOLO polygon labels. 1,258 defective images are balanced with 1,258 good images
and split 70 / 15 / 15 (stratified by category and defect type).

In [ ]:
# ============================================================
# BUILD FULL 15-CATEGORY YOLO SEGMENTATION DATASET
# ============================================================

import cv2
import yaml
import shutil
import random
import numpy as np

from pathlib import Path
from collections import defaultdict
from sklearn.model_selection import train_test_split


# ============================================================
# SETTINGS
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

ALL_CATEGORIES = [
    "bottle",
    "cable",
    "capsule",
    "carpet",
    "grid",
    "hazelnut",
    "leather",
    "metal_nut",
    "pill",
    "screw",
    "tile",
    "toothbrush",
    "transistor",
    "wood",
    "zipper"
]

YOLO_DIR_15 = Path(
    "/kaggle/working/mvtec_yolo_seg_15cat"
)

print("=" * 70)
print("BUILD SEGMENTATION DATASET")
print("=" * 70)

print("MVTec path :", MVTEC_PATH)
print("YOLO path  :", YOLO_DIR_15)


# ============================================================
# 1. COLLECT ALL DEFECTIVE IMAGES + MASKS
# ============================================================

all_defective_samples = []
missing_masks = []

for category in ALL_CATEGORIES:

    category_path = MVTEC_PATH / category

    test_dir = category_path / "test"
    gt_dir = category_path / "ground_truth"

    defect_types = sorted([
        p.name
        for p in test_dir.iterdir()
        if p.is_dir() and p.name != "good"
    ])

    for defect_type in defect_types:

        image_dir = test_dir / defect_type
        mask_dir = gt_dir / defect_type

        for image_path in sorted(
            image_dir.glob("*.png")
        ):

            mask_path = (
                mask_dir /
                f"{image_path.stem}_mask.png"
            )

            if not mask_path.exists():

                missing_masks.append(
                    str(mask_path)
                )

                continue

            all_defective_samples.append({
                "category": category,
                "defect_type": defect_type,
                "image": image_path,
                "mask": mask_path
            })


print("\nDefective images :", len(all_defective_samples))
print("Missing masks    :", len(missing_masks))

assert len(all_defective_samples) == 1258
assert len(missing_masks) == 0


# ============================================================
# 2. COLLECT TRAIN/GOOD
# ============================================================

good_by_category = defaultdict(list)

for category in ALL_CATEGORIES:

    good_dir = (
        MVTEC_PATH /
        category /
        "train" /
        "good"
    )

    good_by_category[category] = sorted(
        good_dir.glob("*.png")
    )


total_available_good = sum(
    len(v)
    for v in good_by_category.values()
)

print(
    "Available train/good:",
    total_available_good
)

assert total_available_good == 3629


# ============================================================
# 3. SELECT 1,258 GOOD IMAGES
# ============================================================

TARGET_GOOD = len(
    all_defective_samples
)

base_per_category = (
    TARGET_GOOD //
    len(ALL_CATEGORIES)
)

remainder = (
    TARGET_GOOD %
    len(ALL_CATEGORIES)
)

selected_good_samples = []

random.seed(SEED)

for i, category in enumerate(
    ALL_CATEGORIES
):

    target = (
        base_per_category +
        (1 if i < remainder else 0)
    )

    available = (
        good_by_category[category]
    )

    n_select = min(
        target,
        len(available)
    )

    selected = random.sample(
        available,
        n_select
    )

    for image_path in selected:

        selected_good_samples.append({
            "category": category,
            "defect_type": "good",
            "image": image_path,
            "mask": None
        })


# Fill any shortage
selected_paths = {
    str(s["image"])
    for s in selected_good_samples
}

remaining_needed = (
    TARGET_GOOD -
    len(selected_good_samples)
)

if remaining_needed > 0:

    remaining_pool = []

    for category in ALL_CATEGORIES:

        for image_path in (
            good_by_category[category]
        ):

            if (
                str(image_path)
                not in selected_paths
            ):

                remaining_pool.append(
                    (category, image_path)
                )

    extra = random.sample(
        remaining_pool,
        remaining_needed
    )

    for category, image_path in extra:

        selected_good_samples.append({
            "category": category,
            "defect_type": "good",
            "image": image_path,
            "mask": None
        })


print(
    "Selected good     :",
    len(selected_good_samples)
)

assert len(
    selected_good_samples
) == 1258


# ============================================================
# 4. SPLIT DEFECTIVE — 70 / 15 / 15
# ============================================================

defective_strata = [
    f"{s['category']}__{s['defect_type']}"
    for s in all_defective_samples
]

def_train, def_temp = train_test_split(
    all_defective_samples,
    test_size=0.30,
    random_state=SEED,
    stratify=defective_strata
)

temp_strata = [
    f"{s['category']}__{s['defect_type']}"
    for s in def_temp
]

def_val, def_test = train_test_split(
    def_temp,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_strata
)


# ============================================================
# 5. SPLIT GOOD — 70 / 15 / 15
# ============================================================

good_strata = [
    s["category"]
    for s in selected_good_samples
]

good_train, good_temp = train_test_split(
    selected_good_samples,
    test_size=0.30,
    random_state=SEED,
    stratify=good_strata
)

good_temp_strata = [
    s["category"]
    for s in good_temp
]

good_val, good_test = train_test_split(
    good_temp,
    test_size=0.50,
    random_state=SEED,
    stratify=good_temp_strata
)


# ============================================================
# 6. COMBINE
# ============================================================

train_samples_15 = (
    def_train +
    good_train
)

val_samples_15 = (
    def_val +
    good_val
)

test_samples_15 = (
    def_test +
    good_test
)


print("\nSplit:")

print(
    f"Train: {len(train_samples_15)} "
    f"({len(def_train)} defective + "
    f"{len(good_train)} good)"
)

print(
    f"Val  : {len(val_samples_15)} "
    f"({len(def_val)} defective + "
    f"{len(good_val)} good)"
)

print(
    f"Test : {len(test_samples_15)} "
    f"({len(def_test)} defective + "
    f"{len(good_test)} good)"
)


# ============================================================
# 7. CHECK 73 DEFECT TYPES
# ============================================================

all_types = {
    f"{s['category']}/{s['defect_type']}"
    for s in all_defective_samples
}

train_types = {
    f"{s['category']}/{s['defect_type']}"
    for s in def_train
}

val_types = {
    f"{s['category']}/{s['defect_type']}"
    for s in def_val
}

test_types = {
    f"{s['category']}/{s['defect_type']}"
    for s in def_test
}


print("\nDefect types:")
print("All   :", len(all_types))
print("Train :", len(train_types))
print("Val   :", len(val_types))
print("Test  :", len(test_types))

assert len(all_types) == 73
assert len(train_types) == 73
assert len(val_types) == 73
assert len(test_types) == 73


# ============================================================
# 8. CREATE KAGGLE YOLO DIRECTORY
# ============================================================

if YOLO_DIR_15.exists():

    shutil.rmtree(
        YOLO_DIR_15
    )


for split in [
    "train",
    "val",
    "test"
]:

    (
        YOLO_DIR_15 /
        "images" /
        split
    ).mkdir(
        parents=True,
        exist_ok=True
    )

    (
        YOLO_DIR_15 /
        "labels" /
        split
    ).mkdir(
        parents=True,
        exist_ok=True
    )


# ============================================================
# 9. BINARY MASK -> YOLO SEGMENTATION POLYGONS
# ============================================================

def mask_to_yolo_labels(
    mask_path,
    class_id=0,
    min_area=10
):

    mask = cv2.imread(
        str(mask_path),
        cv2.IMREAD_GRAYSCALE
    )

    if mask is None:
        return []

    h, w = mask.shape

    _, binary = cv2.threshold(
        mask,
        127,
        255,
        cv2.THRESH_BINARY
    )

    contours, _ = cv2.findContours(
        binary,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    labels = []

    for contour in contours:

        if (
            cv2.contourArea(contour)
            < min_area
        ):
            continue

        epsilon = (
            0.001 *
            cv2.arcLength(
                contour,
                True
            )
        )

        polygon = cv2.approxPolyDP(
            contour,
            epsilon,
            True
        )

        if len(polygon) < 3:
            continue

        polygon = polygon.reshape(
            -1,
            2
        )

        coordinates = []

        for x, y in polygon:

            coordinates.extend([
                np.clip(x / w, 0, 1),
                np.clip(y / h, 0, 1)
            ])

        labels.append(
            [class_id] +
            coordinates
        )

    return labels


# ============================================================
# 10. CONVERT ONE SPLIT
# ============================================================

def convert_split(
    samples,
    split_name
):

    defective = 0
    good = 0
    regions = 0
    failed = 0

    for sample in samples:

        category = sample["category"]
        defect_type = sample["defect_type"]

        image_path = sample["image"]
        mask_path = sample["mask"]

        new_stem = (
            f"{category}_"
            f"{defect_type}_"
            f"{image_path.stem}"
        )

        output_image = (
            YOLO_DIR_15 /
            "images" /
            split_name /
            f"{new_stem}.png"
        )

        output_label = (
            YOLO_DIR_15 /
            "labels" /
            split_name /
            f"{new_stem}.txt"
        )

        shutil.copy2(
            image_path,
            output_image
        )

        # GOOD
        if defect_type == "good":

            output_label.touch()

            good += 1

            continue

        # DEFECTIVE
        labels = mask_to_yolo_labels(
            mask_path
        )

        if len(labels) == 0:

            output_label.touch()

            failed += 1

            continue

        with open(
            output_label,
            "w"
        ) as f:

            for label in labels:

                coords = " ".join(
                    f"{v:.6f}"
                    for v in label[1:]
                )

                f.write(
                    f"{int(label[0])} "
                    f"{coords}\n"
                )

        defective += 1

        regions += len(labels)

    return {
        "total": len(samples),
        "defective": defective,
        "good": good,
        "regions": regions,
        "failed": failed
    }


# ============================================================
# 11. CONVERT
# ============================================================

random.seed(SEED)

random.shuffle(
    train_samples_15
)

random.shuffle(
    val_samples_15
)

random.shuffle(
    test_samples_15
)


print("\nConverting images...")


train_stats = convert_split(
    train_samples_15,
    "train"
)

val_stats = convert_split(
    val_samples_15,
    "val"
)

test_stats = convert_split(
    test_samples_15,
    "test"
)


# ============================================================
# 12. CREATE data.yaml
# ============================================================

data_yaml = {
    "path": str(YOLO_DIR_15),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {
        0: "defect"
    }
}

DATA_YAML = (
    YOLO_DIR_15 /
    "data.yaml"
)

with open(
    DATA_YAML,
    "w"
) as f:

    yaml.safe_dump(
        data_yaml,
        f,
        sort_keys=False
    )


# ============================================================
# 13. FINAL CHECK
# ============================================================

print("\n" + "=" * 70)
print("DATASET READY")
print("=" * 70)

for name, stats in [
    ("TRAIN", train_stats),
    ("VAL", val_stats),
    ("TEST", test_stats)
]:

    print(f"\n{name}")

    print(
        "Images     :",
        stats["total"]
    )

    print(
        "Defective  :",
        stats["defective"]
    )

    print(
        "Good       :",
        stats["good"]
    )

    print(
        "Regions    :",
        stats["regions"]
    )

    print(
        "Failed     :",
        stats["failed"]
    )


# ============================================================
# 14. ASSERTIONS
# ============================================================

assert train_stats["total"] == 1760
assert val_stats["total"] == 378
assert test_stats["total"] == 378

assert train_stats["defective"] == 880
assert val_stats["defective"] == 189
assert test_stats["defective"] == 189

assert train_stats["good"] == 880
assert val_stats["good"] == 189
assert test_stats["good"] == 189

assert train_stats["failed"] == 0
assert val_stats["failed"] == 0
assert test_stats["failed"] == 0


print("\nDataset :", YOLO_DIR_15)
print("YAML    :", DATA_YAML)

print("\nALL CHECKS PASSED")

## 4. Train the defect segmentation model

Two stages:

1. **Base training** — YOLOv8s-seg from COCO-pretrained weights.
2. **Fine-tuning** — continues from the base weights with stronger lighting / colour augmentation for better robustness.

In [ ]:
from ultralytics import YOLO
from pathlib import Path

RUNS_DIR = Path("/kaggle/working/runs")
RUNS_DIR.mkdir(parents=True, exist_ok=True)

base_model = YOLO("yolov8s-seg.pt")

base_results = base_model.train(
    data=str(DATA_YAML),
    epochs=80,
    imgsz=640,
    batch=16,
    device=DEVICE,
    optimizer="AdamW",
    lr0=0.001,
    patience=15,
    close_mosaic=10,
    workers=4,
    project=str(RUNS_DIR),
    name="defect_seg_base",
    exist_ok=True,
    save_period=5,
    seed=42,
    deterministic=True,
    plots=True,
    verbose=True,
)

BASE_BEST = Path(base_results.save_dir) / "weights" / "best.pt"
print("Base model:", BASE_BEST, "| exists:", BASE_BEST.exists())

In [ ]:
finetune_model = YOLO(str(BASE_BEST))

finetune_results = finetune_model.train(
    data=str(DATA_YAML),
    epochs=30,
    imgsz=640,
    batch=16,
    device=DEVICE,
    optimizer="AdamW",
    lr0=0.0001,
    lrf=0.01,
    weight_decay=0.0005,
    patience=10,

    # Lighting / colour augmentation
    hsv_h=0.025,
    hsv_s=0.45,
    hsv_v=0.45,

    # Mild geometric augmentation
    degrees=5.0,
    translate=0.08,
    scale=0.20,
    shear=2.0,
    fliplr=0.5,
    flipud=0.0,

    mosaic=0.25,
    mixup=0.0,
    close_mosaic=5,

    project=str(RUNS_DIR),
    name="defect_seg_finetune",
    exist_ok=True,
    seed=42,
    deterministic=True,
    plots=True,
    verbose=True,
)

SEG_BEST = Path(finetune_results.save_dir) / "weights" / "best.pt"
print("Segmentation model:", SEG_BEST, "| exists:", SEG_BEST.exists())

## 5. Calibrate the confidence threshold (validation set only)

An image is flagged as defective when its highest detection confidence reaches the threshold.
Rule: among thresholds with **recall ≥ 95 %**, choose the one with the best F1 (ties broken by specificity).
The test set is **not** used here.

In [ ]:
import gc
import numpy as np
from ultralytics import YOLO
from pathlib import Path

seg_model = YOLO(str(SEG_BEST))
INFER_DEVICE = 0 if torch.cuda.is_available() else "cpu"

def image_level_scores(split):
    """Highest detection confidence per image + ground-truth defective flag."""
    image_dir = YOLO_DIR_15 / "images" / split
    label_dir = YOLO_DIR_15 / "labels" / split
    images = sorted(image_dir.glob("*.png"))

    y_true = np.array([
        int((label_dir / f"{p.stem}.txt").stat().st_size > 0) for p in images
    ])

    scores = []
    for p in images:
        r = seg_model.predict(source=str(p), imgsz=640, conf=0.001, iou=0.7,
                              device=INFER_DEVICE, verbose=False)[0]
        scores.append(float(r.boxes.conf.max()) if r.boxes is not None and len(r.boxes) else 0.0)
        del r
    gc.collect()
    return y_true, np.array(scores)

def qc_metrics(y_true, scores, threshold):
    y_pred = (scores >= threshold).astype(int)
    TP = int(((y_true == 1) & (y_pred == 1)).sum())
    TN = int(((y_true == 0) & (y_pred == 0)).sum())
    FP = int(((y_true == 0) & (y_pred == 1)).sum())
    FN = int(((y_true == 1) & (y_pred == 0)).sum())
    precision = TP / (TP + FP) if TP + FP else 0.0
    recall = TP / (TP + FN) if TP + FN else 0.0
    specificity = TN / (TN + FP) if TN + FP else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return dict(threshold=float(threshold), TP=TP, TN=TN, FP=FP, FN=FN,
                accuracy=(TP + TN) / len(y_true), precision=precision,
                recall=recall, specificity=specificity, f1=f1)

y_val, s_val = image_level_scores("val")
print("Validation images:", len(y_val), "| defective:", int(y_val.sum()))

rows = [qc_metrics(y_val, s_val, t) for t in np.arange(0.005, 0.501, 0.005)]
eligible = [r for r in rows if r["recall"] >= 0.95]

if eligible:
    best = max(eligible, key=lambda r: (r["f1"], r["specificity"]))
    rule = "recall >= 95%, best F1"
else:
    best = max(rows, key=lambda r: (r["f1"], r["specificity"]))
    rule = "no threshold reached 95% recall; best F1 used"

CONF_THRESHOLD = round(best["threshold"], 3)

print(f"\nSelected threshold: {CONF_THRESHOLD}  ({rule})")
for k in ("accuracy", "precision", "recall", "specificity", "f1"):
    print(f"  {k:<12}: {best[k] * 100:.2f}%")

## 6. Final test evaluation (threshold locked)

In [ ]:
y_test, s_test = image_level_scores("test")
test_qc = qc_metrics(y_test, s_test, CONF_THRESHOLD)

print("=" * 60)
print(f"IMAGE-LEVEL QUALITY CONTROL — TEST SET (conf = {CONF_THRESHOLD})")
print("=" * 60)
print(f"Images : {len(y_test)}   Defective: {int(y_test.sum())}")
print(f"TP={test_qc['TP']}  TN={test_qc['TN']}  FP={test_qc['FP']}  FN={test_qc['FN']}")
for k in ("accuracy", "precision", "recall", "specificity", "f1"):
    print(f"{k:<12}: {test_qc[k] * 100:.2f}%")

seg_metrics = seg_model.val(data=str(DATA_YAML), split="test", imgsz=640, conf=0.001,
                            device=INFER_DEVICE, verbose=False)

print("\nSegmentation metrics (test)")
print(f"{'':<10}{'P':>8}{'R':>8}{'mAP50':>8}{'mAP50-95':>10}")
print(f"{'Box':<10}{seg_metrics.box.mp:>8.3f}{seg_metrics.box.mr:>8.3f}{seg_metrics.box.map50:>8.3f}{seg_metrics.box.map:>10.3f}")
print(f"{'Mask':<10}{seg_metrics.seg.mp:>8.3f}{seg_metrics.seg.mr:>8.3f}{seg_metrics.seg.map50:>8.3f}{seg_metrics.seg.map:>10.3f}")

## 7. Product classifier

A YOLO11s-cls model recognises which of the 15 products is shown. It is used to verify that the uploaded image matches
the category selected by the user before defect inspection starts.

In [ ]:
from pathlib import Path
import shutil
import random
from collections import defaultdict

# ============================================================
# VISIONGUARD PRODUCT CLASSIFIER
# BUILD 15-CLASS DATASET
# ============================================================

SEED = 42
random.seed(SEED)

CLASSIFIER_PATH = Path("/kaggle/working/visionguard_product_classifier")

PRODUCT_CLASSES = [
    "bottle",
    "cable",
    "capsule",
    "carpet",
    "grid",
    "hazelnut",
    "leather",
    "metal_nut",
    "pill",
    "screw",
    "tile",
    "toothbrush",
    "transistor",
    "wood",
    "zipper",
]

# Start clean
if CLASSIFIER_PATH.exists():
    shutil.rmtree(CLASSIFIER_PATH)

for split in ["train", "val", "test"]:
    for category in PRODUCT_CLASSES:
        (CLASSIFIER_PATH / split / category).mkdir(
            parents=True,
            exist_ok=True
        )

split_stats = defaultdict(dict)

print("=" * 75)
print("VISIONGUARD — BUILDING PRODUCT CLASSIFICATION DATASET")
print("=" * 75)

for category in PRODUCT_CLASSES:

    category_path = MVTEC_PATH / category

    images = []

    # --------------------------------------------------------
    # Collect TRAIN images
    # --------------------------------------------------------
    train_path = category_path / "train"

    if train_path.exists():
        images.extend(list(train_path.rglob("*.png")))

    # --------------------------------------------------------
    # Collect TEST images
    # good + defective
    # --------------------------------------------------------
    test_path = category_path / "test"

    if test_path.exists():
        images.extend(list(test_path.rglob("*.png")))

    images = sorted(images)

    # Shuffle reproducibly
    random.Random(SEED).shuffle(images)

    n = len(images)

    # --------------------------------------------------------
    # 70 / 15 / 15 split
    # --------------------------------------------------------
    n_train = int(n * 0.70)
    n_val = int(n * 0.15)

    train_images = images[:n_train]
    val_images = images[n_train:n_train + n_val]
    test_images = images[n_train + n_val:]

    splits = {
        "train": train_images,
        "val": val_images,
        "test": test_images,
    }

    # --------------------------------------------------------
    # Copy images
    # --------------------------------------------------------
    for split_name, split_images in splits.items():

        output_dir = CLASSIFIER_PATH / split_name / category

        for i, src in enumerate(split_images):

            # Unique name because train/test may contain same filenames
            new_name = (
                f"{category}_{i:04d}_"
                f"{src.parent.name}_{src.name}"
            )

            dst = output_dir / new_name

            shutil.copy2(src, dst)

        split_stats[category][split_name] = len(split_images)

    print(
        f"{category:<15} | "
        f"Train: {len(train_images):>3} | "
        f"Val: {len(val_images):>3} | "
        f"Test: {len(test_images):>3} | "
        f"Total: {n:>3}"
    )


# ============================================================
# FINAL VALIDATION
# ============================================================

print("\n" + "=" * 75)
print("FINAL DATASET SUMMARY")
print("=" * 75)

grand_train = 0
grand_val = 0
grand_test = 0

for category in PRODUCT_CLASSES:

    train_count = len(
        list((CLASSIFIER_PATH / "train" / category).glob("*"))
    )

    val_count = len(
        list((CLASSIFIER_PATH / "val" / category).glob("*"))
    )

    test_count = len(
        list((CLASSIFIER_PATH / "test" / category).glob("*"))
    )

    grand_train += train_count
    grand_val += val_count
    grand_test += test_count

    print(
        f"{category:<15} | "
        f"{train_count:>3} | "
        f"{val_count:>3} | "
        f"{test_count:>3}"
    )


print("-" * 75)

print(f"TRAIN : {grand_train}")
print(f"VAL   : {grand_val}")
print(f"TEST  : {grand_test}")
print(f"TOTAL : {grand_train + grand_val + grand_test}")

print("-" * 75)

print("Dataset path:")
print(CLASSIFIER_PATH)

print("=" * 75)


# ============================================================
# SAFETY CHECKS
# ============================================================

assert grand_train + grand_val + grand_test == 5354

for category in PRODUCT_CLASSES:

    assert split_stats[category]["train"] > 0
    assert split_stats[category]["val"] > 0
    assert split_stats[category]["test"] > 0

print("✅ ALL CHECKS PASSED")
print("✅ 15 PRODUCT CLASSES READY FOR TRAINING")

In [ ]:
# ============================================================
# VISIONGUARD PRODUCT CLASSIFIER
# TRAIN 15-CLASS PRODUCT RECOGNITION MODEL
# ============================================================

from ultralytics import YOLO
import torch
from pathlib import Path

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

CLASSIFIER_DATA = Path(
    "/kaggle/working/visionguard_product_classifier"
)

RUNS_PATH = "/kaggle/working/runs"

RUN_NAME = "VisionGuard_ProductClassifier_15Class"

# ------------------------------------------------------------
# GPU Check
# ------------------------------------------------------------

print("=" * 70)
print("VISIONGUARD PRODUCT CLASSIFIER")
print("=" * 70)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU count:", torch.cuda.device_count())

    for i in range(torch.cuda.device_count()):
        print(
            f"GPU {i}:",
            torch.cuda.get_device_name(i)
        )

print("-" * 70)

print("Dataset:")
print(CLASSIFIER_DATA)

print("-" * 70)

# ------------------------------------------------------------
# Load pretrained classification model
# ------------------------------------------------------------

model = YOLO("yolo11s-cls.pt")

print("Base model loaded: YOLO11s-cls")
print("Target classes: 15")

print("=" * 70)

# ------------------------------------------------------------
# Training
# ------------------------------------------------------------

results = model.train(

    data=str(CLASSIFIER_DATA),

    # Training
    epochs=50,
    imgsz=224,
    batch=64,

    # GPU
    device=0 if torch.cuda.is_available() else "cpu",

    # Optimizer
    optimizer="AdamW",
    lr0=0.0005,
    lrf=0.01,
    weight_decay=0.0005,

    # Early stopping
    patience=10,

    # --------------------------------------------------------
    # Augmentation
    # Important for product recognition + class imbalance
    # --------------------------------------------------------

    hsv_h=0.015,
    hsv_s=0.30,
    hsv_v=0.30,

    degrees=5.0,
    translate=0.05,
    scale=0.15,

    fliplr=0.5,
    flipud=0.0,

    # --------------------------------------------------------
    # Reproducibility
    # --------------------------------------------------------

    seed=42,
    deterministic=True,

    # --------------------------------------------------------
    # Output
    # --------------------------------------------------------

    project=RUNS_PATH,
    name=RUN_NAME,
    exist_ok=True,

    save=True,
    plots=True,
    verbose=True
)

# ------------------------------------------------------------
# Best model
# ------------------------------------------------------------

BEST_MODEL = (
    Path(RUNS_PATH)
    / RUN_NAME
    / "weights"
    / "best.pt"
)

print()
print("=" * 70)
print("TRAINING FINISHED")
print("=" * 70)

print("Best model:")
print(BEST_MODEL)

print("Exists:", BEST_MODEL.exists())

print("=" * 70)

### Classifier test evaluation

In [ ]:
import numpy as np
from ultralytics import YOLO
from pathlib import Path
from sklearn.metrics import classification_report, accuracy_score

cls_model = YOLO(str(BEST_MODEL))

test_root = CLASSIFIER_PATH / "test"
y_true, y_pred = [], []

for class_idx, name in cls_model.names.items():
    for img in sorted((test_root / name).glob("*.png")):
        r = cls_model.predict(source=str(img), imgsz=224, verbose=False)[0]
        y_true.append(name)
        y_pred.append(cls_model.names[int(r.probs.top1)])

print(f"Test images: {len(y_true)}")
print(f"Accuracy   : {accuracy_score(y_true, y_pred) * 100:.2f}%\n")
print(classification_report(y_true, y_pred, digits=4))

## 8. Export

Writes the three files the backend needs and zips them for download.

In [ ]:
import json, shutil
from pathlib import Path

EXPORT_DIR = Path("/kaggle/working/export")
if EXPORT_DIR.exists():
    shutil.rmtree(EXPORT_DIR)
EXPORT_DIR.mkdir(parents=True)

shutil.copy2(SEG_BEST, EXPORT_DIR / "defect_segmentation.pt")
shutil.copy2(BEST_MODEL, EXPORT_DIR / "product_classifier.pt")

with open(EXPORT_DIR / "model_config.json", "w") as f:
    json.dump({"confidence_threshold": CONF_THRESHOLD}, f, indent=2)

zip_path = shutil.make_archive("/kaggle/working/visionguard_models", "zip", EXPORT_DIR)

print("Exported to:", EXPORT_DIR)
for p in sorted(EXPORT_DIR.iterdir()):
    print(f"  {p.name:<28}{p.stat().st_size / 1024**2:>8.2f} MB")
print("\nDownload:", zip_path)
print("Copy the three files into backend/model/ and restart the API.")